In [2]:
import pandas as pd  # 데이터프레임(표 형태의 데이터)을 다루기 위한 라이브러리 
import numpy as np   # 수학적 계산 및 무작위 수(난수) 생성을 위한 라이브러리 
import random        
# 무작위 선택을 위한 파이썬 표준 라이브러리 
import boto3         
# AWS(아마존 웹 서비스) 연동용  
# 1. 난수 생성을 위한 시드(Seed) 고정 (매번 실행할 때마다 똑같은 무작위 데이터가 생성되도록 함) 
np.random.seed(42) 
random.seed(42) 
# 가상 데이터 생성에 사용할 장르와 영화 제목 목록 
korean_genres = ['액션', '드라마', '코미디', '로맨스', '스릴러', '판타지', '공포', '애니메이션', '가족', '다큐멘터리'] 
movie_titles = [
    '국가대표', '탐정', '괴물', '도둑들', '공조', '검사외전', '내 친구의 집은 어디인가', 
    '군함도', '암살', '베테랑', '터널', '골든슬럼버', '국가대표2', '대호', '밀정', 
    '특전사령부', '범죄도시', '마약왕', '신의 한수', '나쁜 녀석들', '스윗홈', '나이브스 아웃', 
    '악인전', '혹성탈출', '전쟁 영화', '한산', '명량', '적의', '공중도둑', '신세계', 
    '연평해전', '타짜', '혜화역 1번 출구', '원더', '럭키', '캐스팅', '살인', '계획', 
    '복수는 나의 것', '엄마는 외계인', '아저씨', '곡성', '살인자의 기억법', '라이언 고슬링', 
    '미션임파서블', '분노의 질주', '스타워즈', '어벤져스', '반도', '스파이더맨', '배트맨', 
    '조커', '더 라이트하우스', '컨테이전', '블랙팬서', '캡틴마블', '어벤져스엔드게임' 
] 
 
# 2. 영화 데이터셋을 무작위로 만드는 함수 (기본값: 1000개 영화) 
def generate_movie_dataset(n_movies=1000): 
    # 생성된 데이터들을 저장할 빈 딕셔너리(각 열이 될 리스트들) 
    data = { 
        'title': [], 'genre': [], 'budget': [], 'revenue': [], 
        'director_popularity': [], 'cast_popularity': [], 'runtime': [], 
        'release_month': [], 'vote_average': [], 'production_company_size': [], 
        'marketing_score': [], 'sequel_flag': [], 'hit_10M': [] 
    } 
     
    # 지정한 영화 개수(n_movies)만큼 반복하며 한 줄씩 데이터를 만듭니다. 
    for i in range(n_movies): 
        # [영화 제목 설정] 영화 제목 목록을 순환하며 중복될 경우 뒤에 ' 2', ' 3' 번호를 붙임 
        title_idx = i % len(movie_titles) 
        if i >= len(movie_titles): 
            title = f"{movie_titles[title_idx]} {i // len(movie_titles) + 1}" 
        else: 
            title = movie_titles[title_idx] 
             
        # [장르 설정] 무작위로 하나의 장르 선택 
        genre = random.choice(korean_genres) 
         
        # [제작비(budget) 설정] 로그정규분포를 활용해 생성하고, 최소 10억~최대 500억 사이로 제한 (소수점 첫째자리 반올림) 
        budget = np.random.lognormal(mean=np.log(50), sigma=1.0) 
        budget = round(max(10, min(500, budget)), 1) 
         
        # [감독 및 배우 인지도 설정] 평균 50, 표준편차 25인 정규분포로 생성 후 0~100 사이로 제한 
        director_popularity = round(max(0, min(100, np.random.normal(50, 25))), 1) 
        cast_popularity = round(max(0, min(100, np.random.normal(50, 25))), 1) 
         
        # [상영시간(runtime) 설정] 평균 115분, 표준편차 20분인 정규분포로 생성 후 90~180분 사이로 제한 
        runtime = max(90, min(180, int(np.random.normal(115, 20)))) 
         
        # [개봉월(release_month) 설정] 1월부터 12월 중 무작위 정수 하나 선택 
        release_month = random.randint(1, 12) 
         
        # [평점(vote_average) 설정] 평균 6.5, 표준편차 1.5인 정규분포로 생성 후 1~10점 사이로 제한 
        vote_average = round(max(1.0, min(10.0, np.random.normal(6.5, 1.5))), 1) 
         
        # [제작사 규모 설정] 1(소형), 2(중형), 3(대형) 중 하나를 무작위 선택 
        production_company_size = random.choice([1, 2, 3]) 
         
        # [마케팅 점수 설정] 0~100점 사이로 제한된 인지도 점수 생성 
        marketing_score = round(max(0, min(100, np.random.normal(50, 25))), 1) 
         
        # [속편 여부(sequel_flag) 설정] 25% 확률로 속편(1), 아니면 (0) 
        sequel_flag = 1 if random.random() < 0.25 else 0 
         
        # [성공 공식 계산] 각 요소에 가중치를 곱해 기본 성공 점수 계산 (총합 1.0 만점) 
        success_score = ( 
            (budget / 500) * 0.25 +                  # 제작비 비중 25% 
            (director_popularity / 100) * 0.25 +     # 감독 인지도 비중 25% 
            (cast_popularity / 100) * 0.20 +         # 배우 인지도 비중 20% 
            (vote_average / 10) * 0.15 +             # 평점 비중 15% 
            (marketing_score / 100) * 0.15           # 마케팅 비중 15% 
        ) 
         
        # 성공 점수에 약간의 무작위 노이즈를 섞어서 최종 성공 확률(0.05 ~ 0.95 사이)을 만듭니다. 
        success_probability = min(0.95, max(0.05, success_score + np.random.normal(0, 0.1))) 
         
        # 속편이거나, 성수기(7, 8, 12월)에 개봉하면 성공 확률을 가산합니다. 
        if sequel_flag == 1: 
            success_probability = min(0.95, success_probability + 0.15) 
        if release_month in [7, 8, 12]: 
            success_probability = min(0.95, success_probability + 0.1) 
             
        #최종 계산된 확률에 따라 천만 관객 돌파 여부(1 또는 0)를 결정합니다. 
        hit_10M = 1 if random.random() < success_probability else 0 
         
        # [관객수 및 매출액(revenue) 결정] 천만 돌파 여부에 따라 관객수를 무작위로 부여합니다. 
        if hit_10M == 1: 
            audience = np.random.normal(loc=15_000_000, scale=5_000_000) 
            audience = max(10_000_000, audience) # 천만 돌파 영화는 무조건 1000만 명 이상으로 고정 
        else: 
            audience = np.random.normal(loc=5_000_000, scale=2_000_000) 
            audience = max(100_000, min(9_999_999, audience)) # 실패 영화는 10만~999만 명 사이 
             
        # 매출액 계산: 대략 [관객수 * 12,000원]을 계산한 후 백만 원 단위(1,000,000)로 변환해 소수점 첫째자리까지 저장 
        revenue = round(audience * 12 / 1_000_000, 1) 
         
        # 생성된 한 편의 영화 데이터를 처음에 만든 딕셔너리의 리스트들에 각각 추가(append)합니다. 
        data['title'].append(title) 
        data['genre'].append(genre) 
        data['budget'].append(budget) 
        data['revenue'].append(revenue) 
        data['director_popularity'].append(director_popularity) 
        data['cast_popularity'].append(cast_popularity) 
        data['runtime'].append(runtime) 
        data['release_month'].append(release_month) 
        data['vote_average'].append(vote_average) 
        data['production_company_size'].append(production_company_size) 
        data['marketing_score'].append(marketing_score) 
        data['sequel_flag'].append(sequel_flag) 
        data['hit_10M'].append(hit_10M) 
         
    # 완성된 딕셔너리 데이터를 판다스의 DataFrame(표) 구조로 만들어 반환합니다. 
    return pd.DataFrame(data) 
 
# 3. 이상치(Outlier) 전처리 함수 
def preprocess_data(df): 
    # 데이터프레임에서 숫자형(Numeric) 열들만 골라냅니다. 
    numeric_columns = df.select_dtypes(include=[np.number]).columns 
     
    # 각 숫자형 열들을 돌며 사분위수(IQR) 기준을 벗어나는 극단적인 값들을 한계값으로 조절(Clip)합니다. 
    for col in numeric_columns:
        q1 = df[col].quantile(0.25)  # 하위 25% 지점의 값 
        q3 = df[col].quantile(0.75)  # 상위 75% 지점의 값 
        iqr = q3 - q1                # 그 사이의 범위값(IQR) 
         
        lower = q1 - 1.5 * iqr       # 정상 범위의 하한선 
        upper = q3 + 1.5 * iqr       # 정상 범위의 상한선 
         
        # 하한선보다 작으면 하한선 값으로, 상한선보다 크면 상한선 값으로 깎아냅니다. 
        df[col] = df[col].clip(lower=lower, upper=upper) 
    return df 
 
# --- 메인 실행 흐름 --- 
 
# 1) 1000개의 가상 영화 데이터를 생성합니다. 
df = generate_movie_dataset(1000) 
 
# 2) 극단적인 수치 데이터를 다듬는 전처리를 수행합니다. 
df = preprocess_data(df) 
 
# 3) 'movie_box_office_prediction.csv' 파일로 저장합니다. (한글 깨짐 방지를 위해 utf-8-sig 사용) 
file_name = "movie_box_office_prediction.csv" 
df.to_csv(file_name, index=False, encoding="utf-8-sig") 
 
# 4) 생성된 데이터의 상위 5개 줄을 출력합니다. 
print("--- 데이터 상위 5개 행 보기 ---") 
print(df.head()) 
 
# 5) 천만 영화(1)와 실패 영화(0)의 개수를 각각 카운트하여 출력합니다. 
print("\n--- 천만 돌파 여부 빈도수 구하기 ---") 
print(df["hit_10M"].value_counts().sort_index()) 
 
# 6) 전체 데이터 중 천만 영화가 차지하는 비중(평균)을 백분율(%)로 출력합니다. 
print(f"\n천만 관객 돌파율: {(df['hit_10M'] == 1).mean() * 100:.2f}%") 

--- 데이터 상위 5개 행 보기 ---
  title  genre  budget  revenue  director_popularity  cast_popularity  \
0  국가대표    드라마    82.2    226.0                 46.5             66.2   
1    탐정    드라마    31.3    146.3                 63.6             38.4   
2    괴물     공포    18.2    120.0                 57.9             27.3   
3   도둑들  다큐멘터리    29.0    104.5                 52.8             21.2   
4    공조     가족    49.3    191.8                 23.6             70.6   

   runtime  release_month  vote_average  production_company_size  \
0      145              1           6.1                        3   
1      105             11           6.9                        3   
2       90              1           8.7                        1   
3      122              1           5.6                        3   
4       90              7           6.8                        1   

   marketing_score  sequel_flag  hit_10M  
0             44.1            0        1  
1              2.2            0        1  
